# CUDA Python Lab 01 — Applied Massive Parallelism

**Student:** RASSUL KANATBEK  
**Student ID:** 240103142

Open this notebook in Google Colab, select **Runtime → Change runtime type → T4 GPU**, then run the cells in order. Hardware telemetry, timings, race-condition outputs, and the verification token are intentionally generated by the allocated GPU at run time; they cannot be filled truthfully in advance.

The out-of-bounds experiment is deliberately defective and may invalidate the current CUDA context. Capture its behavior, then restart the Colab runtime before continuing if later CUDA calls fail.


In [1]:
!nvidia-smi

Thu Oct  8 07:18:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from numba import cuda
import numpy as np
import os

if not cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab, select a T4 GPU runtime and reconnect.")

device = cuda.get_current_device()
def _text(value):
    return value.decode("utf-8") if isinstance(value, (bytes, bytearray)) else str(value)

print(f"Student Name: RASSUL KANATBEK")
print(f"Student ID: 240103142")
print(f"Device Name: {_text(device.name)}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {getattr(device, 'MAX_THREADS_PER_BLOCK', 'N/A')}")
print(f"Max Block Dimensions: {getattr(device, 'MAX_BLOCK_DIM_X', 'N/A')}, {getattr(device, 'MAX_BLOCK_DIM_Y', 'N/A')}, {getattr(device, 'MAX_BLOCK_DIM_Z', 'N/A')}")
print(f"Warp Size: {getattr(device, 'WARP_SIZE', 'N/A')}")


Student Name: RASSUL KANATBEK
Student ID: 240103142
Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


## Section 1 — Hardware telemetry

Copy the driver version, CUDA version, GPU name, and VRAM total from the `nvidia-smi` output above. Record compute capability and max threads per block from the next output. These values must come from your assigned Colab instance.


## Section 2 — CPU/GPU crossover benchmark

This follows the manual's pure-Python CPU loop and separates GPU roundtrip time from kernel-only time. CPU timing is skipped above 1,000,000 elements, as in the supplied harness; those rows are marked as not measured and are excluded from the crossover calculation.


In [3]:
import time
from numba import cuda

STUDENT_ID = "240103142"

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

def benchmark_run(N, student_id_seed=int(STUDENT_ID)):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float("nan")

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms


In [4]:
array_sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
benchmark_rows = []
for N in array_sizes:
    cpu_ms, gpu_total_ms, gpu_kernel_ms = benchmark_run(N)
    if np.isfinite(cpu_ms):
        winner = "CPU" if cpu_ms < gpu_total_ms else "GPU"
    else:
        winner = "Not measured"
    benchmark_rows.append({
        "N": N,
        "CPU (ms)": cpu_ms,
        "GPU total (ms)": gpu_total_ms,
        "GPU kernel only (ms)": gpu_kernel_ms,
        "Winner": winner,
    })

print(f"{'N':>12} | {'CPU ms':>12} | {'GPU total ms':>14} | {'GPU kernel ms':>14} | Winner")
print("-" * 78)
for row in benchmark_rows:
    cpu = f"{row['CPU (ms)']:.6f}" if np.isfinite(row["CPU (ms)"]) else "not measured"
    print(f"{row['N']:>12,} | {cpu:>12} | {row['GPU total (ms)']:>14.6f} | {row['GPU kernel only (ms)']:>14.6f} | {row['Winner']}")

crossovers = [r["N"] for r in benchmark_rows
              if np.isfinite(r["CPU (ms)"]) and r["GPU total (ms)"] < r["CPU (ms)"]]
CROSSOVER_N = min(crossovers) if crossovers else None
if CROSSOVER_N is None:
    print("Empirical crossover N*: not reached among rows with measured CPU time.")
else:
    print(f"Empirical crossover N*: {CROSSOVER_N:,}")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


           N |       CPU ms |   GPU total ms |  GPU kernel ms | Winner
------------------------------------------------------------------------------
         100 |     0.089814 |       5.070178 |       0.127410 | CPU
       1,000 |     0.521756 |       2.793848 |       0.129357 | CPU
      10,000 |     8.028768 |       1.981641 |       0.131321 | GPU
     100,000 |   156.659803 |       3.223293 |       0.111318 | GPU
   1,000,000 |   545.329851 |       4.817727 |       0.110975 | GPU
   5,000,000 | not measured |      12.248385 |       0.265156 | Not measured
  10,000,000 | not measured |      24.643106 |       0.447482 | Not measured
Empirical crossover N*: 10,000


**Q1:** Use the printed empirical crossover N*. If it was not reached, report that it was not reached among the sizes with measured CPU time.  
**Q2:** At N = 100, the GPU total includes host-to-device transfer, kernel launch/scheduling, synchronization, and device-to-host transfer. For such a small input, the actual arithmetic is tiny, so fixed transfer and launch latency can dominate; the kernel-only column isolates the much smaller compute portion. The measured columns show the instance-specific proportions.


## Section 3.1 — Missing bounds guard

Run the following cell only for the required anomaly observation. The launch has 1,024 threads for a 1,000-element allocation. An illegal device access can invalidate the current CUDA context; if that occurs, record the exact message and restart the runtime before continuing.


In [5]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # Deliberately missing idx < d_arr.size guard, as specified by the lab.
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on a 1000-element array...")
try:
    defective_kernel[blocks_per_grid, threads_per_block](d_arr)
    cuda.synchronize()
    print("Execution finished without a reported launch error. Checking host copy...")
    h_result = d_arr.copy_to_host()
    print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")
except Exception as exc:
    print(f"Observed exception: {type(exc).__name__}: {exc}")
    print("Record this exact output in the task sheet. Restart the runtime if CUDA is no longer usable.")


Launching defective kernel with 1024 threads on a 1000-element array...
Execution finished without a reported launch error. Checking host copy...
Elements processed: 1000. Last element: 999.0


## Section 3.2 — Thread coordinate inspector

For N = 37 and block size 8, ceiling division gives 5 blocks and 40 launched threads: 37 active and 3 guarded idle threads.


In [6]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
launched_threads = blocks_per_grid * threads_per_block
log_b = cuda.device_array(launched_threads, dtype=np.int32)
log_t = cuda.device_array(launched_threads, dtype=np.int32)
log_g = cuda.device_array(launched_threads, dtype=np.int32)
log_a = cuda.device_array(launched_threads, dtype=np.int32)
inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()
log_b, log_t, log_g, log_a = (x.copy_to_host() for x in (log_b, log_t, log_g, log_a))
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<16}")
print("-" * 60)
for i in range(32, 40):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<16}")
print(f"Total blocks: {blocks_per_grid}; launched threads: {launched_threads}; active: {N}; idle guarded: {launched_threads - N}")


Global ID  | Block ID   | Thread ID  | Status          
------------------------------------------------------------
32         | 4          | 0          | ACTIVE          
33         | 4          | 1          | ACTIVE          
34         | 4          | 2          | ACTIVE          
35         | 4          | 3          | ACTIVE          
36         | 4          | 4          | ACTIVE          
37         | 4          | 5          | IDLE (GUARDED)  
38         | 4          | 6          | IDLE (GUARDED)  
39         | 4          | 7          | IDLE (GUARDED)  
Total blocks: 5; launched threads: 40; active: 37; idle guarded: 3


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## Section 4 — Race condition and atomic reduction

The naive result is intentionally nondeterministic. Each trial resets the output scalar. Run after restarting the runtime if the defective-kernel experiment left the CUDA context in an error state.


In [7]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N_SUM = 50_000
h_ones = np.ones(N_SUM, dtype=np.float32)
d_ones = cuda.to_device(h_ones)
threads = 256
blocks = (N_SUM + threads - 1) // threads

# Compile both kernels before timing.
for kernel in (naive_sum_kernel, atomic_sum_kernel):
    d_tmp = cuda.to_device(np.zeros(1, dtype=np.float32))
    kernel[blocks, threads](d_ones, d_tmp)
    cuda.synchronize()

def run_sum_once(kernel):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    t0 = time.perf_counter()
    kernel[blocks, threads](d_ones, d_total)
    cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    result = float(d_total.copy_to_host()[0])
    return result, elapsed_ms

naive_results = []
naive_times = []
print("Trial | Target | Naive output | Lost updates (target - output)")
for trial in range(1, 6):
    value, elapsed = run_sum_once(naive_sum_kernel)
    naive_results.append(value)
    naive_times.append(elapsed)
    print(f"{trial:>5} | 50000.0 | {value:>12.1f} | {50000.0 - value:>12.1f}")

atomic_value, atomic_time = run_sum_once(atomic_sum_kernel)
print(f"Atomic sum output: {atomic_value:.1f}; equals 50,000.0: {atomic_value == 50000.0}")
print(f"Naive kernel time (trial mean): {np.mean(naive_times):.6f} ms")
print(f"Atomic kernel time: {atomic_time:.6f} ms")


Trial | Target | Naive output | Lost updates (target - output)
    1 | 50000.0 |          2.0 |      49998.0
    2 | 50000.0 |          2.0 |      49998.0
    3 | 50000.0 |          2.0 |      49998.0
    4 | 50000.0 |          2.0 |      49998.0
    5 | 50000.0 |          2.0 |      49998.0
Atomic sum output: 50000.0; equals 50,000.0: True
Naive kernel time (trial mean): 0.120530 ms
Atomic kernel time: 0.293087 ms


**Explanation:** atomic addition makes each read-modify-write indivisible, preventing lost updates. That coordination can serialize competing updates to the same address, reducing throughput. Record the observed runtime; the penalty depends on the GPU and scheduling.


## Section 5 — 2D radial vignette


In [8]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1],
)
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Grid dimensions: X={blocks_2d[0]}, Y={blocks_2d[1]}, total blocks={blocks_2d[0] * blocks_2d[1]}")
print(f"Generated matrix shape: {result_canvas.shape}")
print(f"Center pixel (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Outer corner pixel (0, 0): {result_canvas[0, 0]:.3f}")


Grid dimensions: X=64, Y=64, total blocks=4096
Generated matrix shape: (1024, 1024)
Center pixel (512, 512): 1.000
Outer corner pixel (0, 0): 0.000


## Section 6 — Student-specific verification checksum

This uses the student ID, the allocated device name, a sample of the generated vignette, and the measured crossover result. If no crossover was found in measured rows, the seed value is `None`; record the result exactly as printed.


In [9]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode("utf-8"))
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode("utf-8")
    h.update(gpu_name)
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode("utf-8"))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

CHECKSUM_TOKEN = generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)


OFFICIAL VERIFICATION CHECKSUM TOKEN: A3003F2AEFED577E


## Transfer results to the supplied task sheet

Use the printed outputs to complete instance-specific fields: `nvidia-smi` telemetry; the seven benchmark rows; the defective-kernel console behavior; five naive-sum trials and atomic timing; and the checksum token. The fixed calculations are 5 blocks / 40 launched / 37 active / 3 idle for Section 3.2, and a 64 × 64 grid / 4,096 total blocks / center 1.000 / corner 0.000 for Section 5.
